# Evaluate best model: Neural Network

In [1]:
# imports
import os
# set this env var to avoid segfaults on Apple Silicon when multiple OpenMP runtimes are loaded (e.g. xgboost + sklearn)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import io
import pickle
import json
import glob
from pathlib import Path
import pandas as pd
import sys

# helpers
sys.path.insert(0, "../../")

from models_helper import ModelTrainer
from dataviz_helper import plot_confusion_matrices


class _CPUUnpickler(pickle.Unpickler):
    """Maps CUDA tensors to CPU during unpickling — needed if the NN was trained on a CUDA machine."""
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu", weights_only=False)
        return super().find_class(module, name)


def load_pkl(path):
    with open(path, "rb") as f:
        return _CPUUnpickler(f).load()

In [2]:
# define directories
DATA_DIR       = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"
MODELS_DIR     = "../../4_modeling/4_5_hyperparameter_tuning/models"
BEST_RUNS_PATH = "../5_1_get_best_runs_wandb/best_by_model_type.json"

# define label columns
LABEL_COLS = [
    "label_ifc_entity",
    "label_predefined_type",
    "label_is_external",
    "label_load_bearing",
]

# load feature names from the best run record
with open(BEST_RUNS_PATH, encoding="utf-8") as f:
    best_runs = json.load(f)

FEATURE_SUBSET = best_runs["plain_neural_network"]["oversampling"]["best_val_f1_macro"]["feature_names"]
print(f"Features ({len(FEATURE_SUBSET)}): {FEATURE_SUBSET}")

Features (73): ['aabb_min_x', 'aabb_min_y', 'aabb_min_z', 'aabb_max_x', 'aabb_max_y', 'aabb_max_z', 'aabb_len_x', 'aabb_len_y', 'aabb_len_z', 'aabb_ratio_z_x', 'aabb_ratio_z_y', 'aabb_ratio_x_y', 'aabb_diagonal', 'aabb_volume', 'geom_volume', 'geom_surface_area', 'geom_projected_area', 'geom_centroid_x', 'geom_centroid_y', 'geom_centroid_z', 'geom_z_min', 'geom_z_max', 'geom_z_range', 'geom_ratio_area_vol', 'geom_compactness', 'geom_layer_count', 'tfbb_extent_0', 'tfbb_extent_1', 'tfbb_extent_2', 'tfbb_volume', 'tfbb_ratio_01', 'tfbb_ratio_02', 'tfbb_ratio_12', 'tfbb_linearity', 'tfbb_planarity', 'tfbb_sphericity', 'tfbb_primary_ax_x', 'tfbb_primary_ax_y', 'tfbb_primary_ax_z', 'topo_vertex_count', 'topo_face_count', 'topo_unique_edge_count', 'topo_euler_characteristic', 'topo_genus', 'topo_max_face_area', 'topo_avg_face_area', 'topo_vertex_edge_ratio', 'topo_connected_components', 'mat_allgemein', 'mat_aluminium', 'mat_backstein', 'mat_bekleidung', 'mat_belag', 'mat_beton', 'mat_dämm',

## 1. Load Data and Model

In [3]:
# load validation dataset
df_val = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

X_val = df_val[FEATURE_SUBSET]
y_val = df_val[LABEL_COLS]

print(f"Validation : {len(X_val):,} rows")

Validation : 8,458 rows


In [ ]:
# pick the best Neural Network model (highest score in filename and locally stored as pickle)
model_files = glob.glob(f"{MODELS_DIR}/best_nn_model_*.pkl")
model_path  = max(model_files, key=lambda p: float(Path(p).stem.split("_")[-1]))

print(f"Loading: {model_path}")
model = load_pkl(model_path)

# NNClassifier remembers _device from training on gpuhub -> force it back to MPS if available or CPU otherwise
if hasattr(model.pipeline, "nn_model"):
    model.pipeline._device  = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    model.pipeline.nn_model = model.pipeline.nn_model.to("mps") if torch.backends.mps.is_available() else model.pipeline.nn_model.to("cpu")

print(f"Model type : {type(model).__name__}")
print(f"Labels     : {model.label_cols}")

Loading: ../../4_modeling/4_5_hyperparameter_tuning/models/best_nn_model_0.6784.pkl
Model type : ModelTrainer
Labels     : ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 2. Validation Set Evaluation

In [5]:
# evaluate on validation set to check if everything is working as expected
val_metrics = model.evaluate(X_val, y_val)
val_metrics

,accuracy,precision,recall,mcc,f1_weighted,f1_macro
label,,,,,,
label_ifc_entity,0.8008,0.8083,0.8008,0.7121,0.7678,0.6246
label_predefined_type,0.5355,0.7253,0.5355,0.4813,0.5551,0.4497
label_is_external,0.8267,0.8291,0.8267,0.7218,0.8243,0.8498
label_load_bearing,0.7803,0.7853,0.7803,0.6719,0.7767,0.7893
mean,0.7358,0.7870,0.7358,0.6468,0.7310,0.6784


In [6]:
# compute confusion matrices for all labels on the validation set
val_cms = model.confusion_matrices(X_val, y_val)

plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400)

In [7]:
# compute confusion matrices for all labels on the validation set and show them normalized (by true label count)
plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400, normalize=True)